In [ ]:
!pip install pandas numpy psycopg2-binary


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import psycopg2

# 1. CONNEXION SÉCURISÉE À POSTGRESQL
# On utilise les mêmes identifiants que dans SQLTools
conn = psycopg2.connect(
    host="localhost",
    database="waxal_data",
    user="postgres",
    password="passer" # CHANGER ICI : Mets le mot de passe de ton Postgres
)

# 2. EXTRACTION VIA LA REQUÊTE SQL (Brique 1)
query = """
SELECT t.agent_id, a.region, a.quartier, t.montant, t.mois
FROM transactions t
JOIN agents a ON t.agent_id = a.agent_id
WHERE t.mois IN (5, 6);
"""

# Chargement direct du résultat SQL dans un DataFrame Pandas
df = pd.read_sql_query(query, conn)
conn.close() # On referme la connexion proprement après extraction

print("=== 1. DONNÉES BRUTES RÉCUPÉRÉES ===")
display(df.head())


# 3. VECTORISATION (Création du Tableau Croisé Dynamique)
# On bascule les mois (5 et 6) en colonnes pour comparer les volumes d'activité
tcd = df.groupby(['agent_id', 'region', 'quartier', 'mois'])['montant'].sum().unstack(fill_value=0)
tcd.columns = ['Volume_Mai', 'Volume_Juin']
tcd = tcd.reset_index()


# 4. SÉCURITÉ ALGORITHMIQUE (Éviter le crash si un volume est à 0)
tcd['Taux_Baisse_Pct'] = np.where(
    tcd['Volume_Mai'] > 0, 
    ((tcd['Volume_Mai'] - tcd['Volume_Juin']) / tcd['Volume_Mai']) * 100, 
    0.0
)


# 5. LOGIQUE MÉTIER FINTECH (Classification des Agents)
def evaluer_statut(baisse):
    if baisse >= 100:
        return "Churn Total (Boutique Fermée)"
    elif baisse >= 30:
        return "Alerte Critique (Danger Churn)"
    elif baisse > 0:
        return "Baisse Légère"
    else:
        return "En Croissance / Stable"

tcd['Statut_Agent'] = tcd['Taux_Baisse_Pct'].apply(evaluer_statut)


# 6. EXPORTATION POUR POWER BI
tcd.to_csv("churn_agents_senegal.csv", index=False)

print("\n=== 2. ANALYSE DU CHURN TERMINÉE (PRÊT POUR POWER BI) ===")
display(tcd)

=== 1. DONNÉES BRUTES RÉCUPÉRÉES ===


C:\Users\Moulaye Idriss\AppData\Local\Temp\ipykernel_6964\3430563531.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,agent_id,region,quartier,montant,mois
0,AG001,Dakar,Medina,400000,6
1,AG001,Dakar,Medina,1500000,5
2,AG002,Dakar,Pikine,850000,6
3,AG002,Dakar,Pikine,800000,5
4,AG003,Thies,Escale,2000000,5



=== 2. ANALYSE DU CHURN TERMINÉE (PRÊT POUR POWER BI) ===


,agent_id,region,quartier,Volume_Mai,Volume_Juin,Taux_Baisse_Pct,Statut_Agent
0,AG001,Dakar,Medina,1500000,400000,73.333333,Alerte Critique (Danger Churn)
1,AG002,Dakar,Pikine,800000,850000,-6.250000,En Croissance / Stable
2,AG003,Thies,Escale,2000000,0,100.000000,Churn Total (Boutique Fermée)
